# Transilien — MAE Challenge (repartir de zéro)
## Améliorations vs test2.ipynb
- **`p0q1_derived`** : retard au stop immédiatement précédent (dérivé de `p0q2` du stop suivant — aucune fuite)
- **`tg_mean / tg_std`** : interaction train × gare (target encoding)
- **`gare_dow_mean/std`** : interaction gare × jour de semaine
- **`tendance_q`, `delta_q12`, `arret_fraction`, `is_weekend`, `month`** : nouvelles features structurelles
- **LightGBM** (regression_l1) à la place de Random Forest
- **NN amélioré** : early stopping sur 5% hold-out pour le re-training final (fini les epochs fixes !)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import copy
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x_train = pd.read_csv('x_train_final.csv')
y_train = pd.read_csv('y_train_final.csv')
x_test  = pd.read_csv('x_test_final.csv')

y = y_train["p0q0"].copy()

print(f"x_train: {x_train.shape}  |  x_test: {x_test.shape}")
print(f"y: mean={y.mean():.3f}, std={y.std():.3f}, min={y.min()}, max={y.max()}")
print(f"Distrib: 0={( y==0).mean():.1%} | ±1={(y.abs()==1).mean():.1%} | abs>1={(y.abs()>1).mean():.1%}")
print(f"Outliers |y|>15: {(y.abs()>15).mean():.4%}")
print(f"\nDevice: {'cuda:0 — '+torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 1. Feature Engineering (amélioré)

In [ ]:
# ===== Concat train + test =====
full = pd.concat([x_train, x_test], axis=0).reset_index(drop=True)
full = full.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")
full["date"] = pd.to_datetime(full["date"])

# ===== Features temporelles =====
full["day_of_week"] = full["date"].dt.dayofweek   # 0=lundi
full["is_weekend"]  = (full["day_of_week"] >= 5).astype(int)
full["month"]       = full["date"].dt.month

# ===== Agrégats p-lags et q-lags =====
cols_plag = ["p2q0", "p3q0", "p4q0"]
cols_qlag = ["p0q2", "p0q3", "p0q4"]
cols_all  = cols_plag + cols_qlag

full["mean_retard_train"]  = full[cols_plag].mean(axis=1)
full["mean_retard_gare"]   = full[cols_qlag].mean(axis=1)
full["mean_retard_global"] = full[cols_all].mean(axis=1)
full["std_retard_train"]   = full[cols_plag].std(axis=1)
full["std_retard_gare"]    = full[cols_qlag].std(axis=1)
full["tendance_train"]     = full["p2q0"] - full["p4q0"]   # tendance p-lag
full["tendance_q"]         = full["p0q2"] - full["p0q4"]   # NEW: tendance q-lag
full["max_retard"]         = full[cols_all].max(axis=1)
full["min_retard"]         = full[cols_all].min(axis=1)

# ===== NEW: p0q1_derived — retard au stop immédiatement précédent =====
# Astuce sans fuite : p0q1(arret i) = p0q2(arret i+1)  [même train, même date]
# car p0q2(arret i+1) = p0q0(arret i+1 - 2) = p0q0(arret i-1) = p0q1(arret i)
lookup_q1 = full[["train", "date", "arret", "p0q2"]].copy()
lookup_q1["arret"] = lookup_q1["arret"] - 1          # décale: fournit q1 pour l'arrêt courant
lookup_q1 = lookup_q1.rename(columns={"p0q2": "p0q1_derived"})
full = full.merge(lookup_q1[["train", "date", "arret", "p0q1_derived"]],
                  on=["train", "date", "arret"], how="left")
full["p0q1_derived"] = full["p0q1_derived"].fillna(full["p0q2"])       # fallback: p0q2
full["delta_q12"]    = full["p0q1_derived"] - full["p0q2"]             # accélération du retard
print(f"p0q1_derived: {full['p0q1_derived'].notna().mean():.1%} renseigné")

# ===== Position de la gare dans le trajet =====
train_part = full.iloc[:len(y)]
position_gare = {}
for (train_id, date), groupe in train_part.groupby(["train", "date"]):
    groupe = groupe.sort_values("arret")
    n = len(groupe)
    for i, (_, row) in enumerate(groupe.iterrows()):
        position_gare.setdefault(row["gare"], []).append(i / (n - 1) if n > 1 else 0)

position_df = pd.DataFrame({
    "gare": list(position_gare.keys()),
    "position_moyenne": [np.mean(v) for v in position_gare.values()]
})
full = full.merge(position_df, on="gare", how="left")
full["position_moyenne"] = full["position_moyenne"].fillna(0.5)

# arret_fraction = position relative min-max dans le trajet
arret_stats = train_part.groupby(["train", "date"])["arret"].agg(arret_min="min", arret_max="max")
full = full.join(arret_stats, on=["train", "date"])
full["arret_fraction"] = ((full["arret"] - full["arret_min"]) /
                           (full["arret_max"] - full["arret_min"]).replace(0, 1))
full["arret_fraction"] = full["arret_fraction"].fillna(0.5)
full = full.drop(columns=["arret_min", "arret_max"])

# ===== Fréquence de trains par gare =====
freq_gare = train_part.groupby(["gare", "date"])["train"].nunique().groupby("gare").mean().reset_index()
freq_gare.columns = ["gare", "freq_trains_par_jour"]
full = full.merge(freq_gare, on="gare", how="left")
full["freq_trains_par_jour"] = full["freq_trains_par_jour"].fillna(0)

# ===== NEW: Gare × Day of Week (stats target, train seulement) =====
gare_dow_df = pd.DataFrame({
    "gare":        x_train.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")["gare"].values,
    "day_of_week": pd.to_datetime(x_train["date"]).dt.dayofweek.values,
    "p0q0":        y.values
})
gare_dow_stats = gare_dow_df.groupby(["gare", "day_of_week"])["p0q0"].agg(
    gare_dow_mean="mean", gare_dow_std="std"
).reset_index()
full = full.merge(gare_dow_stats, on=["gare", "day_of_week"], how="left")
full["gare_dow_mean"] = full["gare_dow_mean"].fillna(0)
full["gare_dow_std"]  = full["gare_dow_std"].fillna(1)

# ===== DiGraph (code prof — inchangé) =====
print("Construction du DiGraph...")
sub_graphs = {}
for day, group_day in train_part.groupby("date"):
    sub_G = nx.DiGraph()
    for _, group_train in group_day.groupby("train"):
        group_train = group_train.sort_values("arret")
        gares     = group_train["gare"].values
        stops     = group_train["arret"].values
        delays    = group_train["p2q0"].values
        delays_s2 = group_train["p0q2"].values
        edges = [(gares[i], gares[i+1]) for i in range(len(gares)-1) if stops[i+1] == stops[i]+1]
        sub_G.add_edges_from(edges)
        for i in range(len(edges)):
            try:
                d = sub_G.edges[edges[i]]
                nx.set_edge_attributes(sub_G, {edges[i]: {
                    "delay":    d["delay"]    + delays[i+1],
                    "count":    d["count"]    + 1,
                    "delay_s2": d["delay_s2"] + delays_s2[i+1]
                }})
            except:
                nx.set_edge_attributes(sub_G, {edges[i]: {
                    "delay": delays[i+1], "count": 1, "delay_s2": delays_s2[i+1]
                }})
    sub_graphs[str(day.date())] = copy.deepcopy(sub_G)

G = nx.DiGraph()
for g in sub_graphs.values():
    G = nx.compose(G, g)

edge_data = {}
for e in G.edges:
    edge_data[e] = {"delay": 0, "count": 0, "delay_s2": 0}
    for g in sub_graphs.values():
        if e in g.edges:
            edge_data[e]["delay"]    += g.edges[e]["delay"]
            edge_data[e]["count"]    += g.edges[e]["count"]
            edge_data[e]["delay_s2"] += g.edges[e]["delay_s2"]
nx.set_edge_attributes(G, edge_data)
del edge_data

for u, v, d in G.edges(data=True):
    d["mean_delay"]    = d["delay"]    / d["count"] if d["count"] > 0 else 0
    d["mean_delay_s2"] = d["delay_s2"] / d["count"] if d["count"] > 0 else 0

gare_in_delay, gare_out_delay, gare_in_delay_s2, gare_out_count = {}, {}, {}, {}
for node in G.nodes():
    in_e  = list(G.in_edges(node,  data=True))
    out_e = list(G.out_edges(node, data=True))
    gare_in_delay[node]    = np.mean([d["mean_delay"]    for _, _, d in in_e])  if in_e  else 0
    gare_out_delay[node]   = np.mean([d["mean_delay"]    for _, _, d in out_e]) if out_e else 0
    gare_in_delay_s2[node] = np.mean([d["mean_delay_s2"] for _, _, d in in_e])  if in_e  else 0
    gare_out_count[node]   = sum(d["count"] for _, _, d in out_e)

pagerank = nx.pagerank(G, weight="count", max_iter=200)
print(f"DiGraph: {G.number_of_nodes()} gares, {G.number_of_edges()} arcs")

full["gare_retard_entrant"]    = full["gare"].map(gare_in_delay).fillna(0)
full["gare_retard_sortant"]    = full["gare"].map(gare_out_delay).fillna(0)
full["gare_retard_entrant_s2"] = full["gare"].map(gare_in_delay_s2).fillna(0)
full["gare_trafic_sortant"]    = full["gare"].map(gare_out_count).fillna(0)
full["gare_pagerank"]          = full["gare"].map(pagerank).fillna(0)

# ===== Label encoding =====
le_train = LabelEncoder();  full["train_enc"] = le_train.fit_transform(full["train"])
le_gare  = LabelEncoder();  full["gare_enc"]  = le_gare.fit_transform(full["gare"])

_date      = full["date"]
_train_str = full["train"]
_gare_str  = full["gare"]
full = full.drop(columns=["date"]).fillna(0)
full["date"]      = _date
full["train_str"] = _train_str
full["gare_str"]  = _gare_str

x_train_fe = full.iloc[:len(y)].copy()
x_test_fe  = full.iloc[len(y):].copy()

print(f"\nx_train_fe: {x_train_fe.shape}  |  x_test_fe: {x_test_fe.shape}")
print("Nouvelles features ajoutées : p0q1_derived, delta_q12, tendance_q,",
      "gare_dow_mean/std, arret_fraction, is_weekend, month")

## 2. Target Encoding + LightGBM (replace Random Forest)

In [ ]:
# ===== Target encoding : rolling 7j VECTORISÉ (30x plus rapide) =====
def compute_stats_7j_fast(fit, group_col, target_col="p0q0"):
    """
    Rolling 7j vectorisé via pivot matriciel.
    Pour chaque date D, calcule mean/std sur (D-7 < date < D) strict.
    """
    fit = fit[["date", group_col, target_col]].copy()
    fit["date"] = pd.to_datetime(fit["date"])
    fit["sq"]   = fit[target_col] ** 2

    daily_agg = fit.groupby([group_col, "date"]).agg(
        s=(target_col, "sum"), sq=("sq", "sum"), n=(target_col, "count")
    ).reset_index()

    s_piv  = daily_agg.pivot(index="date", columns=group_col, values="s").sort_index().fillna(0)
    sq_piv = daily_agg.pivot(index="date", columns=group_col, values="sq").sort_index().fillna(0)
    n_piv  = daily_agg.pivot(index="date", columns=group_col, values="n").sort_index().fillna(0)

    records = []
    for D in sorted(fit["date"].unique()):
        mask = (s_piv.index > D - pd.Timedelta(days=7)) & (s_piv.index < D)
        if not mask.any():
            continue
        sum_w  = s_piv[mask].sum(axis=0)
        sum2_w = sq_piv[mask].sum(axis=0)
        cnt_w  = n_piv[mask].sum(axis=0)
        valid  = cnt_w > 0
        mean_  = (sum_w / cnt_w.where(valid, 1)).where(valid, np.nan)
        var_   = (sum2_w / cnt_w.where(valid, 1) - mean_**2).where(valid, np.nan).clip(lower=0)
        std_   = np.sqrt(var_)
        df_d   = pd.DataFrame({
            group_col:              sum_w.index,
            f"{group_col}_mean_7j": mean_.values,
            f"{group_col}_std_7j":  std_.values,
            "date": D
        })
        records.append(df_d[valid.values])

    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def add_target_features(df, x_fit, y_fit):
    """Target encoding (gare, train, train×gare) + rolling 7j. Calculé sur x_fit/y_fit."""
    result = df.reset_index(drop=True).copy()
    fit    = x_fit[["gare_enc", "train_enc", "gare_str", "train_str", "date"]].copy().reset_index(drop=True)
    fit["p0q0"] = y_fit.values
    fit["date"] = pd.to_datetime(fit["date"])

    # Stats globales par gare
    sg = fit.groupby("gare_enc")["p0q0"].agg(
        gare_mean="mean", gare_std="std", gare_median="median",
        gare_q25=lambda x: x.quantile(0.25), gare_q75=lambda x: x.quantile(0.75)
    ).reset_index()
    result = result.merge(sg, on="gare_enc", how="left")

    # Stats globales par train
    st = fit.groupby("train_enc")["p0q0"].agg(
        train_mean="mean", train_std="std", train_median="median"
    ).reset_index()
    result = result.merge(st, on="train_enc", how="left")

    # Interaction train × gare
    stg = fit.groupby(["train_enc", "gare_enc"])["p0q0"].agg(
        tg_mean="mean", tg_std="std"
    ).reset_index()
    result = result.merge(stg, on=["train_enc", "gare_enc"], how="left")
    result["tg_mean"] = result["tg_mean"].fillna(result["gare_mean"])
    result["tg_std"]  = result["tg_std"].fillna(result["gare_std"])

    # Rolling 7j vectorisé
    result["date"] = pd.to_datetime(result["date"])

    s7j_t = compute_stats_7j_fast(fit.rename(columns={"train_enc": "te_"}), "te_")
    s7j_g = compute_stats_7j_fast(fit.rename(columns={"gare_enc":  "ge_"}), "ge_")

    if not s7j_t.empty:
        s7j_t = s7j_t.rename(columns={"te_": "train_enc", "te__mean_7j": "train_mean_7j", "te__std_7j": "train_std_7j"})
        result = result.merge(s7j_t, on=["train_enc", "date"], how="left")
    else:
        result["train_mean_7j"] = np.nan;  result["train_std_7j"] = np.nan

    if not s7j_g.empty:
        s7j_g = s7j_g.rename(columns={"ge_": "gare_enc", "ge__mean_7j": "gare_mean_7j", "ge__std_7j": "gare_std_7j"})
        result = result.merge(s7j_g, on=["gare_enc", "date"], how="left")
    else:
        result["gare_mean_7j"] = np.nan;  result["gare_std_7j"] = np.nan

    result["train_mean_7j"] = result["train_mean_7j"].fillna(result["train_mean"])
    result["train_std_7j"]  = result["train_std_7j"].fillna(result["train_std"])
    result["gare_mean_7j"]  = result["gare_mean_7j"].fillna(result["gare_mean"])
    result["gare_std_7j"]   = result["gare_std_7j"].fillna(result["gare_std"])

    result = result.drop(columns=["date", "train_str", "gare_str", "train", "gare"], errors="ignore")
    return result


# ===== Split 90/10 + target features =====
x_tr_raw, x_val_raw, y_tr, y_val = train_test_split(
    x_train_fe, y, test_size=0.10, random_state=42
)
import time
t0 = time.time()
print("Computing target features (train fold)...")
x_tr  = add_target_features(x_tr_raw, x_tr_raw, y_tr)
print("Computing target features (val fold)...")
x_val = add_target_features(x_val_raw, x_tr_raw, y_tr)
print(f"Target features done en {time.time()-t0:.0f}s")
print(f"x_tr: {x_tr.shape}  |  x_val: {x_val.shape}")


## 3. Neural Network (amélioré)

In [ ]:
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ===== NN_FEATURES : 31 features — EXACTEMENT test2.ipynb =====
# Diagnostic : v4 (34f) avait val 0.6177 mais leaderboard 0.6736 > 0.6605 (test2)
# Le gap val→LB s'élargissait : les 3 nouvelles features overfittaient l'IID mais pas le temporel.
# Retour strict aux 31 features de test2.ipynb, en gardant seulement y-scaling (vraie amélioration).
# Features retirées vs v4 : p0q1_derived, arret_fraction, gare_dow_mean
NN_FEATURES = [
    "arret",
    "p2q0", "p3q0", "p4q0",
    "p0q2", "p0q3", "p0q4",
    "mean_retard_train", "mean_retard_gare", "mean_retard_global",
    "std_retard_train", "std_retard_gare",
    "tendance_train",
    "max_retard", "min_retard",
    "gare_enc",
    "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
    "position_moyenne", "freq_trains_par_jour",
    "day_of_week",
    "gare_mean_7j", "gare_std_7j",
    "gare_retard_entrant", "gare_retard_sortant",
    "gare_retard_entrant_s2", "gare_trafic_sortant", "gare_pagerank",
]
print(f"NN features : {len(NN_FEATURES)}  (=test2.ipynb exact)")

X_nn_tr  = x_tr[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
X_nn_val = x_val[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)

scaler_X = RobustScaler()
X_nn_tr  = np.clip(scaler_X.fit_transform(X_nn_tr), -10, 10).astype(np.float32)
X_nn_val = np.clip(scaler_X.transform(X_nn_val),    -10, 10).astype(np.float32)

scaler_y = StandardScaler()
y_tr_s   = scaler_y.fit_transform(y_tr.values.reshape(-1, 1)).flatten().astype(np.float32)
y_val_s  = scaler_y.transform(y_val.values.reshape(-1, 1)).flatten().astype(np.float32)
print(f"y scaled range: [{y_tr_s.min():.1f}, {y_tr_s.max():.1f}]")


class MLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(),
            nn.BatchNorm1d(256, momentum=0.01, eps=0.001), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.BatchNorm1d(128, momentum=0.01, eps=0.001), nn.Dropout(0.2),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.BatchNorm1d(64,  momentum=0.01, eps=0.001),
            nn.Linear(64, 32),   nn.ReLU(),
            nn.Linear(32, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_nn(X_train, y_train, X_val, y_val, n_features, seed=42,
             epochs=200, batch_size=1024, patience=30):
    torch.manual_seed(seed);  np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    model     = MLP(n_features).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, eps=1e-7)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=10, min_lr=1e-6
    )
    loss_fn = nn.HuberLoss(delta=1.0)

    X_tr_t  = torch.tensor(X_train, device=device)
    y_tr_t  = torch.tensor(y_train, device=device)
    X_val_t = torch.tensor(X_val,   device=device)
    y_val_t = torch.tensor(y_val,   device=device)
    n_s = X_tr_t.shape[0]

    best_mae, best_state, wait = float('inf'), None, 0

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n_s, device=device)
        for i in range(0, n_s, batch_size):
            idx  = perm[i:i+batch_size]
            loss = loss_fn(model(X_tr_t[idx]), y_tr_t[idx])
            optimizer.zero_grad();  loss.backward();  optimizer.step()

        model.eval()
        with torch.no_grad():
            val_mae = (model(X_val_t) - y_val_t).abs().mean().item()
        scheduler.step(val_mae)

        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    model.load_state_dict(best_state);  model.to(device)
    return model, epoch + 1 - patience


N_SEEDS = 5
nn_models, nn_val_preds, nn_stop_epochs = [], [], []

for seed_i in range(N_SEEDS):
    seed = 42 + seed_i * 7
    print(f"\n--- NN seed {seed} ({seed_i+1}/{N_SEEDS}) ---")
    m, stop_ep = train_nn(X_nn_tr, y_tr_s, X_nn_val, y_val_s, len(NN_FEATURES), seed=seed)
    m.eval()
    with torch.no_grad():
        pred_i_s = m(torch.tensor(X_nn_val, device=device)).cpu().numpy()
    pred_i = scaler_y.inverse_transform(pred_i_s.reshape(-1, 1)).flatten()
    mae_i = mean_absolute_error(y_val, pred_i)
    print(f"  MAE val: {mae_i:.4f}  (arrondi: {mean_absolute_error(y_val, np.round(pred_i)):.4f})  stop_ep≈{stop_ep}")
    nn_models.append(m);  nn_val_preds.append(pred_i);  nn_stop_epochs.append(stop_ep)

pred_nn_val = np.mean(nn_val_preds, axis=0)
mae_nn      = mean_absolute_error(y_val, pred_nn_val)
mae_nn_r    = mean_absolute_error(y_val, np.round(pred_nn_val))

print(f"\n{'='*58}")
print(f"MAE NN ensemble (5 seeds) : {mae_nn:.4f}  (arrondi: {mae_nn_r:.4f})")
for i, (p, ep) in enumerate(zip(nn_val_preds, nn_stop_epochs)):
    print(f"  seed {i}: MAE={mean_absolute_error(y_val, p):.4f}  stop_ep={ep}")
print(f"Epochs moyen (→ FIXED_EPOCHS) : {int(np.mean(nn_stop_epochs))}")
print(f"{'='*58}")
print(f"\n→ Si val arrondi ≈ 0.62, le LB devrait être ≤ 0.6605")


## 4. Re-entraînement final sur 100% + Soumission

In [ ]:
# ===== Target features sur 100% du train =====
import time
t0 = time.time()
print("Target features sur 100% des données...")
x_train_final = add_target_features(x_train_fe, x_train_fe, y)
x_test_final  = add_target_features(x_test_fe,  x_train_fe, y)
print(f"done en {time.time()-t0:.0f}s")

# ===== Préparer X/y sur 100% =====
X_full_nn  = x_train_final[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
X_test_nn  = x_test_final[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)

scaler_X_full = RobustScaler()
X_full_nn_s   = np.clip(scaler_X_full.fit_transform(X_full_nn), -10, 10).astype(np.float32)
X_test_nn_s   = np.clip(scaler_X_full.transform(X_test_nn),     -10, 10).astype(np.float32)

scaler_y_full = StandardScaler()
y_full_s = scaler_y_full.fit_transform(y.values.reshape(-1, 1)).flatten().astype(np.float32)

# ===== FIXED_EPOCHS = moyenne des early-stop epochs de la validation =====
# Même technique que test2.ipynb : pas de hold-out, entraînement sur 100% pour le nb moyen d'epochs
FIXED_EPOCHS = max(int(np.mean(nn_stop_epochs)), 30)
print(f"FIXED_EPOCHS = {FIXED_EPOCHS}  (moy. early-stop: {nn_stop_epochs})")


def train_nn_fixed(X_train, y_train, n_features, seed=42, epochs=100, batch_size=1024):
    """Entraîne sur 100% des données pour un nombre fixe d'epochs (pas de val set)."""
    torch.manual_seed(seed);  np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    mdl       = MLP(n_features).to(device)
    optimizer = torch.optim.Adam(mdl.parameters(), lr=1e-3, eps=1e-7)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=10, min_lr=1e-6
    )
    loss_fn = nn.HuberLoss(delta=1.0)

    X_t = torch.tensor(X_train, device=device)
    y_t = torch.tensor(y_train, device=device)
    n_s = X_t.shape[0]

    for epoch in range(epochs):
        mdl.train()
        perm  = torch.randperm(n_s, device=device)
        total = 0.0
        nb    = 0
        for i in range(0, n_s, batch_size):
            idx  = perm[i:i+batch_size]
            loss = loss_fn(mdl(X_t[idx]), y_t[idx])
            optimizer.zero_grad();  loss.backward();  optimizer.step()
            total += loss.item();  nb += 1
        scheduler.step(total / nb)

    mdl.eval()
    return mdl


N_SEEDS_FINAL = 10
nn_test_preds = []

print(f"\nRe-entraînement {N_SEEDS_FINAL} NNs sur 100% des données ({FIXED_EPOCHS} epochs)...")
for seed_i in range(N_SEEDS_FINAL):
    seed = 42 + seed_i * 7
    print(f"  NN seed {seed} ({seed_i+1}/{N_SEEDS_FINAL})...", end=" ", flush=True)
    mdl = train_nn_fixed(X_full_nn_s, y_full_s, len(NN_FEATURES), seed=seed, epochs=FIXED_EPOCHS)
    with torch.no_grad():
        p_s    = mdl(torch.tensor(X_test_nn_s, device=device)).cpu().numpy()
    p_test = scaler_y_full.inverse_transform(p_s.reshape(-1, 1)).flatten()
    nn_test_preds.append(p_test)
    print("done")

pred_nn_test = np.mean(nn_test_preds, axis=0)

# ===== Soumission =====
sub_nn = pd.DataFrame({"p0q0": np.round(pred_nn_test).astype(int)})
sub_nn.to_csv("submission_nn_v4.csv", index=True)

print(f"\n{'='*60}")
print(f"MAE NN val (arrondi) : {mae_nn_r:.4f}  ← score challenge estimé")
print(f"Stats test : mean={pred_nn_test.mean():.3f}, std={pred_nn_test.std():.3f}")
print(f"→ submission_nn_v4.csv  (34 features, y-scaled, {N_SEEDS_FINAL} seeds, {FIXED_EPOCHS} epochs)")


: 